# Salary Fact Table Loader

This notebook maintains the `warehouse.fact_salary` fact table.

**Purpose**: Track salary information across job postings

**Key Features**:
* One grain: One row per job with salary data
* Salary range metrics (min, max, midpoint)
* Currency and employment type context
* SCD2-aware dimension joins
* Aggregatable metrics for salary analysis

**Sources**: 
* `silver.silver_jobs_current` (salary data)
* `warehouse.dim_job` (job context)
* Other dimension tables

**Target**: `workspace.warehouse.fact_salary`

In [0]:
%sql
-- Extract jobs with salary information
CREATE OR REPLACE TEMP VIEW salary_extract AS
SELECT 
  j.enterprise_job_id,
  j.source_name,
  j.salary_min,
  j.salary_max,
  j.salary_currency,
  j.employment_type_normalized,
  j.posted_at,
  j.updated_at,
  -- Calculate derived metrics
  (COALESCE(j.salary_min, 0) + COALESCE(j.salary_max, 0)) / 2.0 as salary_midpoint,
  COALESCE(j.salary_max, 0) - COALESCE(j.salary_min, 0) as salary_range_width,
  CASE 
    WHEN j.salary_min IS NOT NULL AND j.salary_max IS NOT NULL THEN TRUE
    ELSE FALSE
  END as has_salary_range
FROM workspace.silver.silver_jobs_current j
WHERE j.enterprise_job_id IS NOT NULL
  AND (j.salary_min IS NOT NULL OR j.salary_max IS NOT NULL)
  AND j.is_active = TRUE
  AND j.soft_delete_flag = FALSE;

In [0]:
%sql
-- Normalize salary to USD (simplified - use exchange rate table in production)
CREATE OR REPLACE TEMP VIEW salary_normalized AS
SELECT 
  s.*,
  CASE s.salary_currency
    WHEN 'USD' THEN 1.0
    WHEN 'EUR' THEN 1.10
    WHEN 'GBP' THEN 1.25
    WHEN 'CAD' THEN 0.75
    WHEN 'AUD' THEN 0.65
    WHEN 'INR' THEN 0.012
    ELSE 1.0
  END as usd_exchange_rate,
  s.salary_min * CASE s.salary_currency
    WHEN 'USD' THEN 1.0
    WHEN 'EUR' THEN 1.10
    WHEN 'GBP' THEN 1.25
    WHEN 'CAD' THEN 0.75
    WHEN 'AUD' THEN 0.65
    WHEN 'INR' THEN 0.012
    ELSE 1.0
  END as salary_min_usd,
  s.salary_max * CASE s.salary_currency
    WHEN 'USD' THEN 1.0
    WHEN 'EUR' THEN 1.10
    WHEN 'GBP' THEN 1.25
    WHEN 'CAD' THEN 0.75
    WHEN 'AUD' THEN 0.65
    WHEN 'INR' THEN 0.012
    ELSE 1.0
  END as salary_max_usd,
  s.salary_midpoint * CASE s.salary_currency
    WHEN 'USD' THEN 1.0
    WHEN 'EUR' THEN 1.10
    WHEN 'GBP' THEN 1.25
    WHEN 'CAD' THEN 0.75
    WHEN 'AUD' THEN 0.65
    WHEN 'INR' THEN 0.012
    ELSE 1.0
  END as salary_midpoint_usd
FROM salary_extract s;

In [0]:
%sql
-- Resolve foreign keys using SCD2 time travel
CREATE OR REPLACE TEMP VIEW fact_salary_with_fks AS
SELECT 
  s.*,
  -- Job SK - resolve to current version
  COALESCE(j.job_sk, -1) as job_sk,
  COALESCE(j.company_sk, -1) as company_sk,
  COALESCE(j.location_sk, -1) as location_sk,
  COALESCE(j.sector_sk, -1) as sector_sk,
  -- Role SK
  COALESCE(dr.role_sk, -1) as role_sk,
  -- Source SK
  COALESCE(ds.source_sk, -1) as source_sk,
  -- Snapshot date SK
  CAST(DATE_FORMAT(s.updated_at, 'yyyyMMdd') AS INT) as snapshot_date_sk
FROM salary_normalized s
-- Join to current job version
LEFT JOIN workspace.warehouse.dim_job j
  ON s.enterprise_job_id = j.enterprise_job_id
  AND j.is_current = TRUE
-- Role dimension
LEFT JOIN workspace.warehouse.dim_role dr
  ON j.canonical_role_id = dr.canonical_role_id
-- Source dimension
LEFT JOIN workspace.warehouse.dim_source ds
  ON s.source_name = ds.source_name;

In [0]:
%sql
-- Merge into fact table
MERGE INTO workspace.warehouse.fact_salary target
USING (
  SELECT
    COALESCE(f.fact_salary_sk,
      ROW_NUMBER() OVER (ORDER BY fk.enterprise_job_id) + COALESCE(max_sk, 0)) as fact_salary_sk,
    fk.job_sk,
    fk.company_sk,
    fk.location_sk,
    fk.role_sk,
    fk.sector_sk,
    fk.source_sk,
    fk.snapshot_date_sk,
    fk.salary_min,
    fk.salary_max,
    fk.salary_midpoint,
    fk.salary_currency,
    fk.salary_min_usd,
    fk.salary_max_usd,
    fk.salary_midpoint_usd,
    fk.salary_range_width,
    fk.has_salary_range,
    fk.employment_type_normalized,
    fk.usd_exchange_rate
  FROM fact_salary_with_fks fk
  LEFT JOIN workspace.warehouse.fact_salary f
    ON fk.enterprise_job_id = f.enterprise_job_id
  CROSS JOIN (
    SELECT COALESCE(MAX(fact_salary_sk), 0) as max_sk 
    FROM workspace.warehouse.fact_salary
  ) max_keys
  WHERE fk.job_sk IS NOT NULL AND fk.job_sk != -1
) source
ON target.job_sk = source.job_sk
WHEN MATCHED THEN UPDATE SET
  target.salary_min = source.salary_min,
  target.salary_max = source.salary_max,
  target.salary_midpoint = source.salary_midpoint,
  target.salary_min_usd = source.salary_min_usd,
  target.salary_max_usd = source.salary_max_usd,
  target.salary_midpoint_usd = source.salary_midpoint_usd,
  target.salary_range_width = source.salary_range_width,
  target.snapshot_date_sk = source.snapshot_date_sk
WHEN NOT MATCHED THEN INSERT (
  fact_salary_sk,
  job_sk,
  company_sk,
  location_sk,
  role_sk,
  sector_sk,
  source_sk,
  snapshot_date_sk,
  salary_min,
  salary_max,
  salary_midpoint,
  salary_currency,
  salary_min_usd,
  salary_max_usd,
  salary_midpoint_usd,
  salary_range_width,
  has_salary_range,
  employment_type_normalized,
  usd_exchange_rate
) VALUES (
  source.fact_salary_sk,
  source.job_sk,
  source.company_sk,
  source.location_sk,
  source.role_sk,
  source.sector_sk,
  source.source_sk,
  source.snapshot_date_sk,
  source.salary_min,
  source.salary_max,
  source.salary_midpoint,
  source.salary_currency,
  source.salary_min_usd,
  source.salary_max_usd,
  source.salary_midpoint_usd,
  source.salary_range_width,
  source.has_salary_range,
  source.employment_type_normalized,
  source.usd_exchange_rate
);

In [0]:
%sql
-- Validate salary fact table
SELECT 
  COUNT(*) as total_salary_records,
  COUNT(DISTINCT job_sk) as jobs_with_salary,
  AVG(salary_midpoint_usd) as avg_salary_usd,
  MIN(salary_min_usd) as min_salary_usd,
  MAX(salary_max_usd) as max_salary_usd,
  SUM(CASE WHEN has_salary_range THEN 1 ELSE 0 END) as jobs_with_range
FROM workspace.warehouse.fact_salary;

-- Salary distribution by role
SELECT 
  r.role_name,
  COUNT(*) as job_count,
  AVG(f.salary_midpoint_usd) as avg_salary,
  MIN(f.salary_min_usd) as min_salary,
  MAX(f.salary_max_usd) as max_salary
FROM workspace.warehouse.fact_salary f
INNER JOIN workspace.warehouse.dim_role r ON f.role_sk = r.role_sk
GROUP BY r.role_name
ORDER BY avg_salary DESC
LIMIT 20;